[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/04_tag_2_3_keras_einstieg_training.ipynb)

# Tag 2/3 - 04 TensorFlow/Keras Einstieg

Keras nimmt uns nicht das Denken über Daten, Modell und Training ab. Es gibt uns aber eine klare Struktur:

1. Daten vorbereiten
2. Modell definieren
3. `compile`: Optimizer, Loss und Metriken setzen
4. `fit`: trainieren
5. `evaluate`: auf ungesehenen Daten prüfen

Das Beispiel bleibt absichtlich klein: zwei Eingabemerkmale, ein Hidden Layer und ein Output-Neuron für binäre Klassifikation.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("TensorFlow:", tf.__version__)


## Daten

`X` enthält zwei Merkmale pro Sample. `y` enthält die Zielklasse: `0` oder `1`.


In [ ]:
def make_binary_data(n=600, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    x0 = rng.normal(loc=(-1.1, -0.7), scale=(0.65, 0.75), size=(n // 2, 2))
    x1 = rng.normal(loc=(1.0, 0.85), scale=(0.7, 0.6), size=(n // 2, 2))
    X = np.vstack([x0, x1]).astype("float32")
    y = np.array([0] * len(x0) + [1] * len(x1), dtype="float32")
    order = rng.permutation(len(y))
    return X[order], y[order]


X, y = make_binary_data()
split = int(0.8 * len(y))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", edgecolor="black", alpha=0.75)
plt.title("Trainingsdaten")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


## Modell

Das Modell besteht aus drei Bausteinen:

- `Input(shape=(2,))`: Jedes Sample hat zwei Zahlen.
- `Dense(8, activation="relu")`: Acht Hidden-Neuronen erzeugen eine nichtlineare Zwischenrepräsentation.
- `Dense(1, activation="sigmoid", name="output")`: Ein Output-Neuron gibt eine Wahrscheinlichkeit zwischen 0 und 1 aus.


In [ ]:
def draw_model_diagram():
    fig, ax = plt.subplots(figsize=(8, 2.2))
    ax.axis("off")
    boxes = [
        (0.08, "Input\n2 Merkmale"),
        (0.39, "Dense\n8 Neuronen\nReLU"),
        (0.70, "output\n1 Neuron\nSigmoid"),
    ]
    for x, label in boxes:
        ax.add_patch(plt.Rectangle((x, 0.35), 0.22, 0.34, fill=False, linewidth=2))
        ax.text(x + 0.11, 0.52, label, ha="center", va="center")
    for x in [0.30, 0.61]:
        ax.annotate("", xy=(x + 0.08, 0.52), xytext=(x, 0.52), arrowprops=dict(arrowstyle="->", linewidth=2))
    ax.set_title("Architektur: kleines neuronales Netz")
    plt.show()


draw_model_diagram()

model = keras.Sequential(
    [
        keras.Input(shape=(2,), name="features"),
        layers.Dense(8, activation="relu", name="hidden_relu"),

        # Wichtig: Das letzte Neuron heißt bewusst output.
        layers.Dense(1, activation="sigmoid", name="output"),
    ]
)

model.summary()


## Compile

`compile` legt fest, wie gelernt wird:

- `optimizer`: Wie werden Gewichte aktualisiert?
- `loss`: Welche Fehlerfunktion wird minimiert?
- `metrics`: Welche Kennzahlen sollen zusätzlich protokolliert werden?


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.03),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)


## Training und Evaluation

`validation_split=0.25` trennt automatisch einen Teil der Trainingsdaten ab. Diese Daten sieht das Modell beim Gewichtsupdate nicht; sie zeigen, ob das Modell generalisiert.


In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.25,
    epochs=35,
    batch_size=32,
    verbose=0,
)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.3f}")
print(f"Test Accuracy: {test_accuracy:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend()
plt.tight_layout()
plt.show()


## Callbacks: Training beobachten und steuern

Ein Callback ist Code, den Keras während `model.fit(...)` automatisch aufruft. Wichtig ist:

**Du rufst die Callback-Funktion nicht selbst auf. Keras ruft sie auf. Deshalb bestimmt Keras auch, welche Input-Parameter möglich sind.**

Es gibt drei Fälle, die man sauber unterscheiden muss.

### 1. Fertige TensorFlow/Keras-Callbacks

TensorFlow/Keras bringt viele Callbacks schon fertig mit. Die liegen in:

`tf.keras.callbacks`

Weil wir oben `from tensorflow.keras import callbacks` importiert haben, schreiben wir kurz:

`callbacks.EarlyStopping(...)`

Beispiele:

- `callbacks.EarlyStopping(...)`: stoppt Training automatisch.
- `callbacks.ModelCheckpoint(...)`: speichert Modellgewichte.
- `callbacks.ReduceLROnPlateau(...)`: reduziert die Lernrate bei Stillstand.
- `callbacks.LearningRateScheduler(...)`: nutzt eine eigene Funktion für die Lernrate.
- `callbacks.TensorBoard(...)`: schreibt Logs für TensorBoard.

Bei diesen fertigen Callbacks sind die erlaubten Parameter im Konstruktor festgelegt. Beispiel:

`callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)`

Diese Werte sind **Konfiguration**:

- `monitor`: Welche Metrik soll beobachtet werden?
- `patience`: Wie lange warten wir ohne Verbesserung?
- `restore_best_weights`: Sollen die besten Gewichte zurückgeladen werden?

Du kannst hier nicht beliebige Parameter erfinden. Erlaubt ist nur, was der jeweilige Callback-Konstruktor kennt.

### 2. Eigene Callback-Klasse

Wenn du selbst einen Callback schreibst, erbst du von:

`callbacks.Callback`

Dann überschreibst du bestimmte Methoden. Auch hier gilt: Die Input-Parameter dieser Methoden sind von Keras fest vorgegeben.

| Methode | Erlaubte Input-Parameter |
|---|---|
| `on_train_begin(self, logs=None)` | `logs` |
| `on_train_end(self, logs=None)` | `logs` |
| `on_epoch_begin(self, epoch, logs=None)` | `epoch`, `logs` |
| `on_epoch_end(self, epoch, logs=None)` | `epoch`, `logs` |
| `on_train_batch_begin(self, batch, logs=None)` | `batch`, `logs` |
| `on_train_batch_end(self, batch, logs=None)` | `batch`, `logs` |
| `on_test_begin(self, logs=None)` | `logs` |
| `on_test_end(self, logs=None)` | `logs` |
| `on_predict_begin(self, logs=None)` | `logs` |
| `on_predict_end(self, logs=None)` | `logs` |

Das ist erlaubt:

`def on_epoch_end(self, epoch, logs=None):`

Das ist nicht erlaubt:

`def on_epoch_end(self, epoch, logs, X_train, y_train):`

Keras weiß nicht, was `X_train` und `y_train` an dieser Stelle sein sollen, weil Keras diese Werte nicht an diesen Hook übergibt.

Wenn dein Callback eigene Zusatzwerte braucht, gibst du sie im Konstruktor `__init__` mit:

`StopAtValidationAccuracy(min_val_accuracy=0.98)`

Im Callback nutzt du dann:

- `logs`: aktuelle Metriken, zum Beispiel `loss`, `accuracy`, `val_loss`, `val_accuracy`.
- `self.model`: das aktuelle Modell, zum Beispiel um `self.model.stop_training = True` zu setzen.
- `self.params`: Informationen zum Trainingslauf, zum Beispiel `epochs` und `steps`.

### 3. Callback mit eigener Funktion

Manche fertigen Callbacks erwarten eine Funktion. Auch dann ist die Signatur fest vorgegeben.

Bei `LearningRateScheduler` muss die Funktion so aussehen:

`def schedule(epoch, lr):`

Keras übergibt automatisch:

- `epoch`: aktuelle Epoche
- `lr`: aktuelle Lernrate

Auch hier gilt: Du kannst nicht einfach weitere Input-Parameter anhängen. Wenn du zusätzliche Werte brauchst, verwendest du eine äußere Variable oder schreibst eine eigene Callback-Klasse.

### Woher kommen die Namen in `logs`?

`logs` ist ein Dictionary. Die Keys entstehen aus `compile(...)` und aus der Validation:

- `loss="binary_crossentropy"` erzeugt `loss`.
- `metrics=["accuracy"]` erzeugt `accuracy`.
- `validation_split` oder `validation_data` erzeugt zusätzlich `val_loss` und `val_accuracy`.
- Manche Callbacks oder Optimizer ergänzen weitere Werte, zum Beispiel `learning_rate`.

Die sicherste Regel ist:

`print(logs.keys())`

oder nach dem Training:

`history.history.keys()`

Genau diese Namen kannst du dann auch als `monitor` verwenden, zum Beispiel `monitor="val_loss"`.


In [ ]:
import inspect


def build_callback_demo_model():
    return keras.Sequential(
        [
            keras.Input(shape=(2,), name="features"),
            layers.Dense(8, activation="relu", name="hidden_relu"),
            layers.Dense(1, activation="sigmoid", name="output"),
        ]
    )


print("Beispiel 1: Fertiger Keras-Callback")
print("EarlyStopping Parameter:")
print(inspect.signature(callbacks.EarlyStopping))
print()

print("Beispiel 2: Callback mit eigener Funktion")
print("LearningRateScheduler Parameter:")
print(inspect.signature(callbacks.LearningRateScheduler))
print()


class CallbackInspector(callbacks.Callback):
    def __init__(self):
        super().__init__()
        self._printed_batch = False

    def on_train_begin(self, logs=None):
        print("\nBeispiel 3: Eigene Callback-Klasse")
        print("on_train_begin bekommt logs.")
        print("Zusätzlich verfügbar: self.params und self.model")
        print("self.params:", self.params)
        print("self.model.name:", self.model.name)

    def on_train_batch_end(self, batch, logs=None):
        # Erlaubte Inputs: batch und logs.
        if not self._printed_batch:
            self._printed_batch = True
            print("\non_train_batch_end bekommt batch und logs.")
            print("batch:", batch)
            print("logs.keys():", sorted((logs or {}).keys()))

    def on_epoch_end(self, epoch, logs=None):
        # Erlaubte Inputs: epoch und logs.
        if epoch == 0:
            print("\non_epoch_end bekommt epoch und logs.")
            print("epoch:", epoch)
            print("logs.keys():", sorted((logs or {}).keys()))


class StopAtValidationAccuracy(callbacks.Callback):
    def __init__(self, min_val_accuracy):
        super().__init__()
        # Eigene Zusatzwerte kommen über __init__, nicht als Extra-Input in on_epoch_end.
        self.min_val_accuracy = min_val_accuracy

    def on_epoch_end(self, epoch, logs=None):
        val_accuracy = (logs or {}).get("val_accuracy")
        if val_accuracy is not None and val_accuracy >= self.min_val_accuracy:
            print(
                f"\nStopAtValidationAccuracy: val_accuracy={val_accuracy:.1%} "
                f">= {self.min_val_accuracy:.1%}. Training wird gestoppt."
            )
            self.model.stop_training = True


def step_schedule(epoch, lr):
    # LearningRateScheduler ruft diese Funktion automatisch als schedule(epoch, lr) auf.
    # Erlaubte Inputs sind hier genau epoch und lr.
    if epoch < 12:
        return 0.03
    return 0.01


tf.keras.utils.set_random_seed(RANDOM_STATE)
callback_model = build_callback_demo_model()
callback_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.03),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stopping = callbacks.EarlyStopping(
    # Fertiger Callback aus tf.keras.callbacks.
    # monitor muss ein Name aus logs oder history.history sein.
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)
lr_scheduler = callbacks.LearningRateScheduler(step_schedule)
stop_at_accuracy = StopAtValidationAccuracy(min_val_accuracy=0.985)

callback_history = callback_model.fit(
    X_train,
    y_train,
    validation_split=0.25,
    epochs=50,
    batch_size=32,
    verbose=0,
    callbacks=[CallbackInspector(), early_stopping, lr_scheduler, stop_at_accuracy],
)

callback_test_loss, callback_test_accuracy = callback_model.evaluate(X_test, y_test, verbose=0)
print()
print("Metriken, die dieses Training erzeugt hat:", list(callback_history.history.keys()))
print(f"EarlyStopping beobachtet: {early_stopping.monitor}")
print(f"Gelaufene Epochen: {len(callback_history.history['loss'])}")
print(f"Test Accuracy mit Callbacks: {callback_test_accuracy:.1%}")

plt.figure(figsize=(7, 4))
plt.plot(callback_history.history["loss"], label="train loss")
plt.plot(callback_history.history["val_loss"], label="validation loss")
plt.title("Callback-Beispiel: EarlyStopping + eigene Callback-Klasse")
plt.xlabel("Epoche")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.7, X[:, 0].max() + 0.7, 160),
    np.linspace(X[:, 1].min() - 0.7, X[:, 1].max() + 0.7, 160),
)
grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
proba = model.predict(grid, verbose=0).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, proba, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.35)
plt.contour(xx, yy, proba, levels=[0.5], colors="black")
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", edgecolor="black")
plt.title("Entscheidungsgrenze des trainierten Keras-Modells")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()
